# CLEF 2026 Task 1 — Retrieve & Rerank Pipeline

**Pipeline:** Dense Retrieval (Harrier) → Cross-Encoder Reranking (Jina) → MRR@5

**Checkpointing:** Each step saves to disk. Re-running skips already-completed steps.

```
Step 1 — Setup & Config
Step 2 — HuggingFace Login
Step 3 — Load Collection & Queries
Step 4 — Load Harrier & define encode()
Step 5 — Embed Documents        → embeddings/{tag}/collection/
Step 6 — Embed Queries          → embeddings/{tag}/{lang}/
Step 7 — Retrieve Top-50        → candidates/{tag}/{lang}_dev.json
Step 8 — Rerank (Jina)          → results/{tag}/{lang}_dev_reranked.json
Step 9 — Evaluate MRR@5
Step 10 — Save Results to Excel
```

## Step 1 — Setup & Config

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import os, json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

langs           = ['en', 'de', 'fr']
TOP_K_RETRIEVE  = 50
TOP_K_RERANK    = 50
TOP_K_EVAL      = 5
RETRIEVER_BATCH = 128
task            = 'Given a web search query, retrieve relevant passages that answer the query'
model_tag       = 'harrier-jina-top50'   # change per experiment
HF_DATASET      = 'sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims'

emb_base  = f'data/embeddings/{model_tag}'
cand_base = f'data/candidates/{model_tag}'
res_base  = f'data/results/{model_tag}'
for d in [f'{emb_base}/collection', cand_base, res_base]:
    os.makedirs(d, exist_ok=True)
for lang in langs:
    os.makedirs(f'{emb_base}/{lang}', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Config ready | device={device} | tag={model_tag}')

✅ Config ready | device=cuda | tag=harrier-jina-top50


## Step 2 — HuggingFace Login

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## Step 3 — Load Collection & Queries

In [3]:
print('🚀 Loading collection...')
collection_data = load_dataset(HF_DATASET, 'collection')['collection']
doc_ids_arr     = np.array(collection_data['pubkey'])

# pubkey → full doc text (used by Jina reranker)
documents = {
    x['pubkey']: f'{x.get("title","")}. {x.get("venue","")}. {x.get("abstract","")}. {x.get("authors","")}'
    for x in collection_data
}

# Add 'passage:' prefix for Harrier document encoding
def create_doc_text(example):
    return {'doc_text': f'passage: {example["title"]}. {example["venue"]}. {example["abstract"]}. {example["authors"]}'}

collection_data = collection_data.map(create_doc_text)
doc_texts = collection_data['doc_text']
print(f'✅ Collection loaded: {len(doc_ids_arr)} documents')

🚀 Loading collection...
✅ Collection loaded: 10000 documents


In [4]:
def instruct(task, q):
    return f'Instruct: {task}\nQuery: {q}'

lang_data = {}
for lang in langs:
    data = load_dataset(HF_DATASET, lang)
    dev  = data['dev']
    lang_data[lang] = {
        'queries':     [instruct(task, t) for t in dev['text']],  # instruct-formatted for Harrier
        'raw_queries': list(dev['text']),                          # plain text for Jina
        'gt':          np.array(dev['pubkey']),
        'index':       list(dev['index']),
    }
    print(f'  {lang}: {len(lang_data[lang]["queries"])} queries')
print('✅ All languages loaded')

  en: 3905 queries
  de: 386 queries
  fr: 702 queries
✅ All languages loaded


## Step 4 — Load Harrier & Define encode()

In [5]:
tokenizer_h = AutoTokenizer.from_pretrained('microsoft/harrier-oss-v1-27b')
model_h     = AutoModel.from_pretrained('microsoft/harrier-oss-v1-27b').to(device)
model_h.eval()
print('✅ Harrier loaded')

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

✅ Harrier loaded


In [6]:
def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_len = attention_mask.sum(dim=1) - 1
    b = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(b, device=last_hidden_states.device), seq_len]

def encode(texts, batch_size=RETRIEVER_BATCH):
    all_embs = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(texts), batch_size), total=total_batches, desc='Encoding'):
        batch_texts = texts[i:i+batch_size]
        batch = tokenizer_h(
            batch_texts, max_length=512, padding=True,
            truncation=True, return_tensors='pt'
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            out = model_h(**batch)
        emb = last_token_pool(out.last_hidden_state, batch['attention_mask'])
        emb = F.normalize(emb, p=2, dim=1)
        all_embs.append(emb.cpu())
        del batch, out, emb
    return torch.cat(all_embs, dim=0).to(device)

## Step 5 — Embed Documents

In [7]:
doc_emb_path = f'{emb_base}/collection/collection_embeddings.pt'
doc_ids_path = f'{emb_base}/collection/doc_ids.npy'

if os.path.exists(doc_emb_path) and os.path.exists(doc_ids_path):
    print('⏭️  Document embeddings found on disk — loading...')
    d_emb       = torch.load(doc_emb_path, map_location=device)
    doc_ids_arr = np.load(doc_ids_path, allow_pickle=True)
else:
    print('🚀 Embedding documents...')
    d_emb = encode(doc_texts)
    torch.save(d_emb.cpu(), doc_emb_path)
    np.save(doc_ids_path, doc_ids_arr)
    print('💾 Collection embeddings saved')

print(f'✅ Document embeddings ready: {d_emb.shape}')

🚀 Embedding documents...


Encoding: 100%|██████████| 79/79 [24:58<00:00, 18.97s/it]


💾 Collection embeddings saved
✅ Document embeddings ready: torch.Size([10000, 5376])


## Step 6 — Embed Queries

In [8]:
lang_embeddings = {}
for lang in langs:
    path = f'{emb_base}/{lang}/{lang}_dev_embeddings.pt'
    if os.path.exists(path):
        print(f'⏭️  {lang} query embeddings found — loading...')
        lang_embeddings[lang] = torch.load(path, map_location=device)
    else:
        print(f'🚀 Embedding {lang} queries...')
        q_emb = encode(lang_data[lang]['queries'])
        torch.save(q_emb.cpu(), path)
        lang_embeddings[lang] = q_emb
        print(f'💾 Saved {lang}_dev_embeddings.pt')
    print(f'✅ {lang}: {lang_embeddings[lang].shape}')
print('\n✅ All query embeddings ready')

🚀 Embedding en queries...


Encoding: 100%|██████████| 31/31 [02:26<00:00,  4.73s/it]


💾 Saved en_dev_embeddings.pt
✅ en: torch.Size([3905, 5376])
🚀 Embedding de queries...


Encoding: 100%|██████████| 4/4 [00:13<00:00,  3.31s/it]


💾 Saved de_dev_embeddings.pt
✅ de: torch.Size([386, 5376])
🚀 Embedding fr queries...


Encoding: 100%|██████████| 6/6 [00:22<00:00,  3.72s/it]

💾 Saved fr_dev_embeddings.pt
✅ fr: torch.Size([702, 5376])

✅ All query embeddings ready


## Step 7 — Retrieve Top-50 (Harrier)

In [10]:
lang_retrieved = {}
for lang in langs:
    cand_path = f'{cand_base}/{lang}_dev.json'
    if os.path.exists(cand_path):
        print(f'⏭️  {lang} candidates found — loading...')
        with open(cand_path, 'r', encoding='utf-8') as f:
            lang_retrieved[lang] = json.load(f)
        continue

    print(f'🚀 Retrieving top-{TOP_K_RETRIEVE} for {lang}...')
    q_emb    = lang_embeddings[lang]
    scores   = (q_emb @ d_emb.T).to(torch.float32).cpu().numpy()
    topk_idx = np.argsort(-scores, axis=1)[:, :TOP_K_RETRIEVE]

    candidates = []
    for i in range(len(lang_data[lang]['queries'])):
        candidates.append({
            'index':       lang_data[lang]['index'][i],
            'text':        lang_data[lang]['raw_queries'][i],
            'gt':          str(lang_data[lang]['gt'][i]),
            'top_pubkeys': doc_ids_arr[topk_idx[i]].tolist(),
        })

    with open(cand_path, 'w', encoding='utf-8') as f:
        json.dump(candidates, f, ensure_ascii=False, indent=2)
    lang_retrieved[lang] = candidates
    print(f'💾 Saved {cand_path} ({len(candidates)} queries)')

print('\n── Retrieval recall ──')
for lang in langs:
    data = lang_retrieved[lang]
    hits = sum(1 for item in data if item['gt'] in item['top_pubkeys'])
    print(f'  {lang}: gold in top-{TOP_K_RETRIEVE}: {hits}/{len(data)} ({100*hits/len(data):.1f}%)')
print('\n✅ Retrieval done')

⏭️  en candidates found — loading...
⏭️  de candidates found — loading...
⏭️  fr candidates found — loading...

── Retrieval recall ──
  en: gold in top-50: 0/3905 (0.0%)
  de: gold in top-50: 0/386 (0.0%)
  fr: gold in top-50: 0/702 (0.0%)

✅ Retrieval done


## Step 8 — Rerank with Jina

In [11]:
# Jina is a large model (~2GB). We only load it if at least one language
# still needs reranking — so resuming after a crash skips the load entirely
# if reranking already finished but e.g. evaluation crashed.
needs_reranking = any(
    not os.path.exists(f'{res_base}/{lang}_dev_reranked.json')
    for lang in langs
)
if needs_reranking:
    print('🚀 Loading Jina reranker...')
    model_j = AutoModel.from_pretrained(
        'jinaai/jina-reranker-v3',
        trust_remote_code=True,
        dtype='auto'
    ).to(device)
    model_j.eval()
    print('✅ Jina reranker loaded')
else:
    print('⏭️  All reranked files found — skipping Jina load')

🚀 Loading Jina reranker...


Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

✅ Jina reranker loaded


In [ ]:
lang_reranked = {}
for lang in langs:
    reranked_path = f'{res_base}/{lang}_dev_reranked.json'
    if os.path.exists(reranked_path):
        print(f'⏭️  {lang} reranked results found — loading...')
        with open(reranked_path, 'r', encoding='utf-8') as f:
            lang_reranked[lang] = json.load(f)
        continue

    print(f'\n🚀 Reranking {lang}...')
    data     = lang_retrieved[lang]
    reranked = []

    for item in tqdm(data, desc=f'{lang} reranking'):
        query  = item['text']
        topk   = item['top_pubkeys'][:TOP_K_RERANK]
        docs   = [documents[p] for p in topk]

        reranked_results = model_j.rerank(query, docs)
        reranked_indices = [r['index'] for r in reranked_results]
        reranked_pubkeys = [topk[idx] for idx in reranked_indices]

        reranked.append({
            'index':            item['index'],
            'text':             query,
            'gt':               item['gt'],
            'reranked_pubkeys': reranked_pubkeys,
        })

    with open(reranked_path, 'w', encoding='utf-8') as f:
        json.dump(reranked, f, ensure_ascii=False, indent=2)
    lang_reranked[lang] = reranked
    print(f'💾 Saved {reranked_path}')

print('\n✅ Reranking done')


🚀 Reranking en...


en reranking:   1%|▏         | 50/3905 [01:31<1:24:50,  1.32s/it]

## Step 9 — Evaluate MRR@5

In [ ]:
results = {}
for lang in langs:
    data = lang_reranked[lang]
    mrr  = 0.0
    for item in tqdm(data, desc=f'MRR@{TOP_K_EVAL} {lang}'):
        top5 = item['reranked_pubkeys'][:TOP_K_EVAL]
        gt   = item['gt']
        if gt in top5:
            mrr += 1.0 / (top5.index(gt) + 1)
    results[lang] = mrr / len(data)
    print(f'✅ {lang} MRR@{TOP_K_EVAL}: {results[lang]:.4f}')

overall = sum(results.values()) / len(results)
print(f"\n{'='*35}")
print(f'  Overall avg MRR@{TOP_K_EVAL}: {overall:.4f}')
print(f"{'='*35}")

## Step 10 — Save Results to Excel

In [ ]:
save_path = 'data/results/results.xlsx'
row = {
    'experiment_name': model_tag,
    'mrr@5_en':        results.get('en'),
    'mrr@5_de':        results.get('de'),
    'mrr@5_fr':        results.get('fr'),
    'mrr@5_avg':       overall,
}
df_new = pd.DataFrame([row])
if os.path.exists(save_path):
    df_old = pd.read_excel(save_path)
    df_new = pd.concat([df_old, df_new], ignore_index=True)
df_new.to_excel(save_path, index=False)
print(f'✅ Results saved to {save_path}')
df_new.tail()